# Проверка SFT: исходная T-lite против T-lite с адаптером (EXP-015)

Решение 03.11.2026 (`docs/decisions.md`, раздел 9): главная проверка SFT — генерация на 25 темах validation в обоих
режимах против исходной T-lite. По одному ответу на тему и режим (T = 0.9 без top-k и top-p, как в RL; seed по теме —
одинаковый у обеих моделей),
плюс по 4 ответа без поиска на 6 темах — разброс награды внутри темы (на нём учится GRPO). Отдельно — loss обеих
моделей на 16 отложенных примерах SFT (точка отсчёта для loss обучения).

Каждая модель — своей ячейкой (`run_model`), сравнение — последней (`compare`); вывод каждой ячейки остаётся.
Карту выбирает аргумент `device`. Эпизоды можно поделить между двумя картами: в копии тетрадки та же модель
с `part=(1, 2)` на другой карте, здесь — `part=(0, 2)`. Готовые эпизоды при перезапуске пропускаются. Результат —
`runs/sft_eval/<RUN>/<модель>/` (эпизоды, ответы, pptx), `runs/sft_eval/<RUN>/comparison.csv` и `episodes.csv`.
В конце — проверка top-k (решение 08.11.2026): SFT-модель с top-k 20 и без на 10 темах × 4 ответа без поиска.

In [ ]:
import collections, csv, dataclasses, gc, json, random, re, statistics, sys, time, traceback, os
from pathlib import Path

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT / "src"))  # если пакет не установлен через pip install -e .
import torch
from presentation_rl.dsl import parse
from presentation_rl.environment import build_prompt, run_episode
from presentation_rl.model_loader import hf_generator, load_model
from presentation_rl.perturb import task_context
from presentation_rl.prompts import system_prompt
from presentation_rl.rendering import build_presentation, pick_theme
from presentation_rl.rewards import (VISUAL_TAU, Lemmatizer, SentenceEmbedder, load_key_points, pairwise_cosine, score,
                                     slide_body, slide_texts)
from presentation_rl.tasks import load_tasks
from presentation_rl.tools.cache import WikiCache
from presentation_rl.tools.wiki import WikiClient
from presentation_rl.tools.wiki_tools import WikiTools
from presentation_rl.training import sft

RUN = "20261107"  # общая папка прогона обеих моделей
config = sft.SFTConfig.load(ROOT / "configs" / "experiments" / "sft_tlite.yaml")  # модель и отложенные темы
SEED = 0
DIVERSITY_TOPICS, DIVERSITY = 6, 4  # темы с тезисами и ответов на тему без поиска — для разброса награды
CONTACT = os.environ.get("WIKI_CONTACT", "https://github.com/doshik722")
# Википедия открыта в пространстве с 10.11.2026; запасной путь — туннель (docs/server.md): "http://127.0.0.1:18080"
os.environ["WIKI_PROXY"] = ""
cache = WikiCache(ROOT / "data" / "sources" / "wiki_cache", WikiClient(contact=CONTACT))
tasks = {t.id: t for t in load_tasks(ROOT / "data" / "tasks" / "tasks.json", split="validation")}
points = load_key_points(ROOT / "data" / "tasks" / "key_points" / "key_points.json")
IDS = sorted(tasks)
WITH_POINTS = [key for key in IDS if key in points]
EPISODES = [(mode, key, 0) for mode in ("search", "plain") for key in IDS]
EPISODES += [("plain", key, s) for key in WITH_POINTS[:DIVERSITY_TOPICS] for s in range(1, DIVERSITY)]
TOPK_TOPICS, TOPK_SAMPLES = 10, 4  # проверка top-k: темы с тезисами и ответов на тему без поиска
TOPK_EPISODES = [("plain", key, s) for key in WITH_POINTS[:TOPK_TOPICS] for s in range(TOPK_SAMPLES)]
print(f"тем {len(IDS)}, с тезисами {len(WITH_POINTS)}; эпизодов на модель {len(EPISODES)}; карт {torch.cuda.device_count()}")

## Функции генерации

`run_model` загружает модель на карту `device` (с адаптером, если он задан), проверяет поиск, считает loss
на отложенных примерах SFT и перплексию на обычном русском тексте — 200 разделах статей Википедии из кеша, проверка
забывания (только при `part[0] == 0`), генерирует свою часть эпизодов и освобождает память.
Эпизод сохраняется сразу: `<режим>_<id>_<номер>.txt` — весь эпизод, `_answer.txt` — ответ, `.json` — время, конец,
вызовы, ошибка; ошибка эпизода прогон не останавливает.

In [ ]:
def heldout_loss(model, tokenizer, prompt):
    """Средний loss по репликам модели на отложенных примерах SFT — как eval_loss при обучении."""
    samples, _ = sft.drop_long(sft.build_samples(ROOT, tokenizer, prompt), config.max_tokens)
    _, held = sft.split(samples, config.eval_tasks, config.seed)
    losses = []
    with torch.no_grad():
        for sample in held:
            inputs = {k: v.to(model.device) for k, v in sft.collate([sample]).items()}
            losses.append(float(sft.masked_loss(model, inputs)))
    return {"examples": len(held), "topics": sorted({s.id for s in held}), "loss": statistics.mean(losses)}


def general_texts(n=200, seed=0, min_words=150, max_words=600):
    """Обычный русский текст для перплексии: разделы статей русской Википедии из кеша (как есть, без шаблона чата)."""
    sections = []
    for line in (ROOT / "data" / "sources" / "wiki_cache" / "articles.jsonl").read_text(encoding="utf-8").splitlines():
        record = json.loads(line)
        if record.get("lang") != "ru":
            continue
        sections += [s["text"] for s in record.get("sections", []) if min_words <= len(s["text"].split()) <= max_words]
    sections = sorted(set(sections))
    return random.Random(seed).sample(sections, min(n, len(sections)))


def general_perplexity(model, tokenizer, texts, max_tokens=512):
    """Перплексия на обычном тексте (проверка забывания): exp среднего loss по всем токенам."""
    total, count = 0.0, 0
    with torch.no_grad():
        for text in texts:
            ids = tokenizer(text, return_tensors="pt", truncation=True, max_length=max_tokens).input_ids.to(model.device)
            loss = model(input_ids=ids, labels=ids).loss
            total += float(loss) * (ids.shape[1] - 1)
            count += ids.shape[1] - 1
    return {"texts": len(texts), "tokens": count, "loss": total / count, "perplexity": float(torch.exp(torch.tensor(total / count)))}


def run_episode_to(out, mode, key, sample, tokenizer, generate, prompt):
    stem = f"{mode}_{key}_{sample}"
    if (out / f"{stem}.json").exists():
        return None
    request = dataclasses.replace(tasks[key].request, search=(mode == "search"))
    start = time.time()
    row = {"mode": mode, "id": key, "sample": sample}
    try:
        tools = WikiTools(cache) if request.search else None
        prompt_ids = prompt(request)
        torch.manual_seed(SEED + 1000 * sample + IDS.index(key))
        episode = run_episode(prompt_ids, tokenizer, generate, tools, max_new_tokens=8000)
        (out / f"{stem}.txt").write_text(tokenizer.decode(episode.ids[len(prompt_ids):]), encoding="utf-8")
        (out / f"{stem}_answer.txt").write_text(episode.answer, encoding="utf-8")
        row.update(finish=episode.finish, generated_tokens=episode.generated_tokens, calls=len(episode.calls), error="")
    except Exception:
        row.update(error=traceback.format_exc()[-1500:])
    row["seconds"] = round(time.time() - start)
    (out / f"{stem}.json").write_text(json.dumps(row, ensure_ascii=False, indent=1), encoding="utf-8")
    return row


def run_model(name, device, adapter=None, part=(0, 1), modes=("search", "plain"), top_k=0, episodes=None, checks=True):
    """Ответы модели ``name`` («base» или «sft») на карте ``device``; ``adapter`` — путь к адаптеру от корня проекта;
    ``modes`` — какие режимы генерировать (без доступа к Википедии — только «plain»); ``top_k`` — 0 (как в RL)
    или 20 (проверка top-k); ``episodes`` — свой список эпизодов вместо ``EPISODES``; ``checks`` — считать loss
    и перплексию."""
    out = ROOT / "runs" / "sft_eval" / RUN / name
    out.mkdir(parents=True, exist_ok=True)
    torch.cuda.set_device(device)
    start = time.time()
    model, tokenizer = load_model(config.model_id, config.revision, dtype="float16", device_map={"": device})
    if adapter:
        from peft import PeftModel

        model = PeftModel.from_pretrained(model, ROOT / adapter)
        model.eval()
    print(f"{name}: загрузка {time.time() - start:.0f} с, память {torch.cuda.memory_allocated(device) / 2**30:.1f} ГБ"
          + (f", адаптер {adapter}" if adapter else "") + f", top-k {top_k}")
    if "search" in modes:  # поиск работает? (ответ из кеша или из Википедии)
        try:
            print(WikiTools(cache).call("search", "рентгеноструктурный анализ ДНК")[:400])
        except Exception as exc:
            print(f"поиск недоступен ({exc}) — эпизоды с поиском пропускаются")
            modes = tuple(m for m in modes if m != "search")

    def prompt(request):
        return build_prompt(tokenizer, system_prompt(request), request)

    if part[0] == 0 and checks:
        loss = {"model": name, "adapter": adapter, **heldout_loss(model, tokenizer, prompt)}
        (out / "heldout_loss.json").write_text(json.dumps(loss, ensure_ascii=False, indent=1), encoding="utf-8")
        print(f"{name}: loss на {loss['examples']} отложенных примерах SFT {loss['loss']:.4f}")
        ppl = {"model": name, **general_perplexity(model, tokenizer, general_texts())}
        (out / "general_perplexity.json").write_text(json.dumps(ppl, ensure_ascii=False, indent=1), encoding="utf-8")
        print(f"{name}: перплексия на {ppl['texts']} разделах русской Википедии {ppl['perplexity']:.3f}")
        torch.cuda.empty_cache()
    generate = hf_generator(model, tokenizer, temperature=0.9, top_k=top_k)
    mine = [e for e in (episodes or EPISODES) if e[0] in modes][part[0]::part[1]]
    for k, (mode, key, sample) in enumerate(mine):
        row = run_episode_to(out, mode, key, sample, tokenizer, generate, prompt)
        if row:
            print(f"{k + 1}/{len(mine)} {mode}_{key}_{sample}: {row.get('finish', '—')}, токенов {row.get('generated_tokens', '—')}, "
                  f"вызовов {row.get('calls', '—')}, {row['seconds']} с" + (" — ОШИБКА" if row["error"] else ""), flush=True)
    del model, generate
    gc.collect()
    torch.cuda.empty_cache()
    print(f"{name}: готово за {(time.time() - start) / 60:.0f} мин")

## Исходная T-lite

Пока на сервере нет доступа к Википедии — только режим без поиска; ответы с поиском — тем же вызовом с
`modes=("search",)`, когда доступ появится (готовые эпизоды пропускаются).

In [ ]:
run_model("base", device=1, modes=("plain",))

## T-lite с адаптером SFT

Адаптер — запуск на 7 эпох, лучшая эпоха 3 (EXP-015).

In [ ]:
run_model("sft", device=1, adapter="runs/sft/20261107_1913/adapter", modes=("plain",))

## Сравнение

Когда готовы ответы обеих моделей. По каждому ответу: доля корректных строк, число слайдов в ожидаемом диапазоне,
заключение последним слайдом, конец ответа, вызовы поиска (с поиском — были ли; без поиска — лишние), `#SOURCE`
из выдачи эпизода; запросы к Википедии (вызовы search, open и read, доля запросов на английском и с найденным,
повторы, доля слайдов со ссылкой на прочитанное); картинки (сколько, длина описания, доля связанных со слайдом —
порог награды для картинок, картинки при запрете); награда и её компоненты. Разница SFT − исходная — по парам «тема, режим» с 95% интервалом
бутстрепа по темам; разброс награды и разнообразие ответов внутри темы (попарное расхождение эмбеддингов текста,
distinct-2) — по дополнительным ответам без поиска.

In [ ]:
CONCLUSION = re.compile(r"^\s*(вывод|выводы|заключение|итог|итоги)", re.IGNORECASE)
RESPONSE = re.compile(r"<tool_response>\n(.*?)\n</tool_response>", re.DOTALL)
CYRILLIC = re.compile("[а-яё]", re.IGNORECASE)


def search_metrics(calls, responses):
    """Запросы к Википедии: сколько вызовов каждого инструмента, доля запросов на английском и с найденным."""
    searches = [(c, r) for c, r in zip(calls, responses + [""] * len(calls)) if c["name"] == "search" and not c["error"]]
    return {"calls_search": len(searches), "calls_open": sum(c["name"] == "open" for c in calls),
            "calls_read": sum(c["name"] == "read" for c in calls),
            "queries_en": statistics.mean(not CYRILLIC.search(c["argument"] or "") for c, _ in searches) if searches else None,
            "queries_found": statistics.mean("ничего не найдено" not in r for _, r in searches) if searches else None}


def image_metrics(deck, embedder):
    """Картинки: сколько, длина описания и доля связанных со слайдом (порог награды для картинок)."""
    images = [(e.data.get("text", ""), slide_body(s, set())) for s in deck.slides for e in s.elements if e.kind == "IMAGE"]
    if not images:
        return {"images": 0, "image_words": None, "images_related": None}
    sims = pairwise_cosine(embedder, [t for t, _ in images], [b for _, b in images])
    return {"images": len(images), "image_words": statistics.mean(len(t.split()) for t, _ in images),
            "images_related": statistics.mean(float(v) >= VISUAL_TAU["IMAGE"] for v in sims)}


def evaluate(name, embedder, lemmatizer):
    folder = ROOT / "runs" / "sft_eval" / RUN / name
    rows = []
    for meta in sorted(folder.glob("*_*_*.json")):
        row = {"model": name, **json.loads(meta.read_text(encoding="utf-8"))}
        rows.append(row)
        if row["error"]:
            continue
        task, stem = tasks[row["id"]], meta.stem
        text = (folder / f"{stem}.txt").read_text(encoding="utf-8")
        context = task_context(task, text, row["mode"] == "search")
        deck = parse((folder / f"{stem}_answer.txt").read_text(encoding="utf-8"))
        report = build_presentation(deck, folder / f"{stem}.pptx", pick_theme(task.request.theme, task.request.text),
                                    sources=(context["sources"] or {}).get, show_sources=task.request.sources)
        n = len(deck.slides)
        statuses = [i.status for i in report.items if i.kind == "SOURCE"]
        row.update(parse_rate=deck.parse_rate, slides=n, stop=row["finish"] == "stop",
                   conclusion=bool(n >= 3 and CONCLUSION.match(deck.slides[-1].title or "")),
                   searched=row["mode"] == "search" and row["calls"] > 0,
                   calls_in_plain=row["mode"] == "plain" and row["calls"] > 0,
                   sources_ok=statuses.count("ok"), sources_bad=statuses.count("bad_source"),
                   slides_with_source=len({i.slide for i in report.items if i.kind == "SOURCE" and i.status == "ok"}) / n
                   if n else 0.0, **image_metrics(deck, embedder))
        if row["mode"] == "search":
            row.update(search_metrics(context["calls"], RESPONSE.findall(text)))
        if row["id"] in points:
            slide_range = tuple(context["slide_range"]) if context["slide_range"] else None
            r = score(deck, points[row["id"]], conclusion=context["conclusion"], slide_range=slide_range,
                      embedder=embedder, lemmatizer=lemmatizer, report=report, images_allowed=context["images_allowed"],
                      tool_text=context["tool_text"], calls=context["calls"], search=context["search"])
            low, high = r.expected_slides
            row.update(in_range=low <= n <= high, **{f"reward_{k}": v for k, v in r.components().items()})
    return rows


def paired(rows, metric, mode, n_boot=2000, seed=0):
    """Среднее у исходной и SFT и разница по парам (тема, режим) с 95% интервалом бутстрепа по темам."""
    get = {(r["model"], r["id"]): r.get(metric) for r in rows if r["mode"] == mode and r["sample"] == 0 and not r["error"]}
    keys = [k for k in IDS if get.get(("base", k)) is not None and get.get(("sft", k)) is not None]
    if not keys:
        return None
    diffs = [float(get[("sft", k)]) - float(get[("base", k)]) for k in keys]
    rng = random.Random(seed)
    boot = sorted(statistics.mean(rng.choice(diffs) for _ in diffs) for _ in range(n_boot))
    return {"mode": mode, "metric": metric, "n": len(keys),
            "base": statistics.mean(float(get[("base", k)]) for k in keys),
            "sft": statistics.mean(float(get[("sft", k)]) for k in keys), "diff": statistics.mean(diffs),
            "low": boot[int(0.025 * n_boot)], "high": boot[int(0.975 * n_boot)]}


def diversity(rows, name, embedder):
    """Разнообразие ответов на одну тему (без поиска, DIVERSITY ответов): среднее попарное 1 − косинус эмбеддингов
    текста слайдов и доля различных пар слов (distinct-2) по всем ответам темы; среднее по темам."""
    folder = ROOT / "runs" / "sft_eval" / RUN / name
    spreads, distinct = [], []
    for key in WITH_POINTS[:DIVERSITY_TOPICS]:
        stems = [f"plain_{key}_{r['sample']}" for r in rows if r["model"] == name and r["mode"] == "plain"
                 and r["id"] == key and not r["error"]]
        texts = []
        for stem in stems:
            deck = parse((folder / f"{stem}_answer.txt").read_text(encoding="utf-8"))
            texts.append("\n".join(t.full for t in slide_texts(deck)))
        if len(texts) < 2:
            continue
        vectors = embedder.encode(texts, "passage")
        pairs = [(i, j) for i in range(len(texts)) for j in range(i + 1, len(texts))]
        spreads.append(statistics.mean(1 - float(vectors[i] @ vectors[j]) for i, j in pairs))
        bigrams = [tuple(w[k:k + 2]) for t in texts for w in [re.findall(r"\w+", t.lower())] for k in range(len(w) - 1)]
        distinct.append(len(set(bigrams)) / max(1, len(bigrams)))
    return {"topics": len(spreads), "embedding_diversity": statistics.mean(spreads) if spreads else None,
            "distinct_2": statistics.mean(distinct) if distinct else None}


def write_csv(path, items):
    fields = list(dict.fromkeys(k for r in items for k in r))
    with open(path, "w", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, fields)
        writer.writeheader()
        writer.writerows(items)


METRICS = ["parse_rate", "slides", "in_range", "conclusion", "stop", "searched", "calls_in_plain", "calls",
           "calls_search", "calls_open", "calls_read", "queries_en", "queries_found", "reward_repeats",
           "sources_ok", "sources_bad", "slides_with_source", "images", "image_words", "images_related",
           "reward_forbidden_images", "generated_tokens", "seconds", "reward_total", "reward_coverage",
           "reward_keys", "reward_length", "reward_visual", "reward_fake_quotes", "reward_format"]


def compare(device=1):
    embedder, lemmatizer = SentenceEmbedder("BAAI/bge-m3", device=f"cuda:{device}"), Lemmatizer()
    rows = evaluate("base", embedder, lemmatizer) + evaluate("sft", embedder, lemmatizer)
    folder = ROOT / "runs" / "sft_eval" / RUN
    write_csv(folder / "episodes.csv", rows)
    for name in ("base", "sft"):
        part = [r for r in rows if r["model"] == name]
        print(f"{name}: эпизодов {len(part)}, ошибок {sum(bool(r['error']) for r in part)}")
        path = folder / name / "heldout_loss.json"
        if path.exists():
            print(f"   loss на отложенных примерах SFT: {json.loads(path.read_text(encoding='utf-8'))['loss']:.4f}")
        path = folder / name / "general_perplexity.json"
        if path.exists():
            print(f"   перплексия на обычном русском тексте: {json.loads(path.read_text(encoding='utf-8'))['perplexity']:.3f}")
    table = [t for mode in ("search", "plain") for metric in METRICS if (t := paired(rows, metric, mode))]
    write_csv(folder / "comparison.csv", table)
    for t in table:
        print(f"{t['mode']:6s} {t['metric']:18s} n={t['n']:2d}  исходная {t['base']:8.3f}  SFT {t['sft']:8.3f}  "
              f"разница {t['diff']:+8.3f}  [{t['low']:+.3f}; {t['high']:+.3f}]")
    for name in ("base", "sft"):  # разброс награды внутри темы (без поиска): на этой разнице учится GRPO
        spreads = []
        for key in WITH_POINTS[:DIVERSITY_TOPICS]:
            values = [r["reward_total"] for r in rows if r["model"] == name and r["mode"] == "plain" and r["id"] == key
                      and not r["error"] and "reward_total" in r]
            if len(values) >= 2:
                spreads.append(statistics.pstdev(values))
        if spreads:
            print(f"разброс награды внутри темы, {name}: {statistics.mean(spreads):.4f} (тем {len(spreads)})")
        d = diversity(rows, name, embedder)
        print(f"разнообразие ответов, {name}: эмбеддинги {d['embedding_diversity']}, distinct-2 {d['distinct_2']} "
              f"(тем {d['topics']})")

In [ ]:
compare(device=1)

## Проверка top-k (решение 08.11.2026)

У T-lite в `generation_config` стоит top-k 20; генератор его не выключал, и пробные ответы SFT получены с ним.
В RL top-k выключен (вероятности при обучении считаются по полному распределению). Без top-k при T = 0.9 исходная
T-lite не вырождалась (EXP-004: 0 из 24; при T = 1 — 4 из 24), но это было до SFT, без DSL и в 4 битах; EXP-004
просил перепроверить после SFT. Здесь SFT-модель на 10 темах × 4 ответа без поиска: top-k 0 и top-k 20, сиды одинаковые.

Признаки вырождения (раньше — только чтением): слова из смеси алфавитов и иероглифы (`broken_words`), самый частый
повтор 8 слов подряд (`loop_8gram`), доля повторённых строк, ответ не закончен. Вырождение — не закончен, или повтор
8 слов ≥ 4 раз, или ≥ 5 испорченных слов. Хвосты отмеченных ответов — в `topk_flagged.txt`, их читаем.

Правило (зафиксировано до запуска): top-k 0 остаётся, если вырожденных ответов при нём не больше, чем при top-k 20,
плюс 2 (из 40), и средняя награда ниже не больше чем на 0,25σ (σ ≈ 0,08). Иначе — T = 0.8 без top-k
(запасной вариант EXP-004; top-k в обучении не используем: у опорной модели в top-k может не оказаться токена политики).

Ответы с top-k 0 пишутся в папку `sft` (часть из них уже есть после ячейки SFT и пропускается), с top-k 20 — в `sft_topk20`.

In [ ]:
run_model("sft", device=1, adapter="runs/sft/20261107_1913/adapter", modes=("plain",), episodes=TOPK_EPISODES, checks=False)

In [ ]:
run_model("sft_topk20", device=1, adapter="runs/sft/20261107_1913/adapter", modes=("plain",), top_k=20,
          episodes=TOPK_EPISODES, checks=False)

In [ ]:
LATIN = re.compile("[a-z]", re.IGNORECASE)
CJK = re.compile("[\u3040-\u30ff\u4e00-\u9fff]")
WORD = re.compile(r"[^\W\d_]+")


def degeneration(answer, finish):
    """Признаки вырождения ответа: испорченные слова (смесь кириллицы и латиницы в одном слове, иероглифы), самый
    частый повтор 8 слов подряд, доля повторённых строк длиннее 20 знаков, ответ не закончен."""
    words = WORD.findall(answer)
    broken = sum((bool(CYRILLIC.search(w)) and bool(LATIN.search(w))) or bool(CJK.search(w)) for w in words)
    tokens = re.findall(r"\w+", answer.lower())
    grams = collections.Counter(tuple(tokens[i:i + 8]) for i in range(len(tokens) - 7))
    loop = max(grams.values(), default=0)
    lines = [line.strip() for line in answer.splitlines() if len(line.strip()) > 20]
    repeated = (len(lines) - len(set(lines))) / len(lines) if lines else 0.0
    unfinished = finish != "stop"
    return {"broken_words": broken, "glitch": broken > 0, "loop_8gram": loop, "repeated_lines": repeated,
            "unfinished": unfinished, "degenerate": unfinished or loop >= 4 or broken >= 5}


def topk_compare(device=1, sigma=0.08, n_boot=2000, seed=0):
    embedder, lemmatizer = SentenceEmbedder("BAAI/bge-m3", device=f"cuda:{device}"), Lemmatizer()
    wanted = {(key, s) for _, key, s in TOPK_EPISODES}
    folder = ROOT / "runs" / "sft_eval" / RUN
    rows, flagged = [], []
    for name, top_k in (("sft", 0), ("sft_topk20", 20)):
        for row in evaluate(name, embedder, lemmatizer):
            if row["mode"] != "plain" or (row["id"], row["sample"]) not in wanted or row["error"]:
                continue
            answer = (folder / name / f"plain_{row['id']}_{row['sample']}_answer.txt").read_text(encoding="utf-8")
            row.update(degeneration(answer, row["finish"]), top_k=top_k)
            rows.append(row)
            if row["degenerate"] or row["glitch"]:
                flagged.append(f"===== top-k {top_k}, {row['id']}_{row['sample']}: испорченных слов {row['broken_words']}, "
                               f"повтор 8 слов {row['loop_8gram']}, не закончен {row['unfinished']}\n…{answer[-800:]}\n")
    write_csv(folder / "topk_check.csv", rows)
    (folder / "topk_flagged.txt").write_text("\n".join(flagged), encoding="utf-8")
    summary = {}
    for top_k in (0, 20):
        part = [r for r in rows if r["top_k"] == top_k]
        summary[top_k] = {"n": len(part), "degenerate": sum(r["degenerate"] for r in part),
                          "glitch": sum(r["glitch"] for r in part), "unfinished": sum(r["unfinished"] for r in part),
                          "loop_8gram": statistics.mean(r["loop_8gram"] for r in part) if part else None,
                          "parse_rate": statistics.mean(r["parse_rate"] for r in part) if part else None,
                          "reward": statistics.mean(r["reward_total"] for r in part) if part else None}
        print(f"top-k {top_k:2d}: " + ", ".join(f"{k} {v:.3f}" if isinstance(v, float) else f"{k} {v}"
                                               for k, v in summary[top_k].items()))
    by_topic = {}  # средняя награда по теме при каждом top-k; разница — по темам, бутстреп по темам
    for r in rows:
        by_topic.setdefault(r["id"], {}).setdefault(r["top_k"], []).append(r["reward_total"])
    diffs = [statistics.mean(v[0]) - statistics.mean(v[20]) for v in by_topic.values() if 0 in v and 20 in v]
    rng = random.Random(seed)
    boot = sorted(statistics.mean(rng.choice(diffs) for _ in diffs) for _ in range(n_boot))
    print(f"награда top-k 0 − top-k 20: {statistics.mean(diffs):+.4f} [{boot[int(0.025 * n_boot)]:+.4f}; "
          f"{boot[int(0.975 * n_boot)]:+.4f}] по {len(diffs)} темам; порог −0,25σ = {-0.25 * sigma:+.4f}")
    keep = (summary[0]["degenerate"] <= summary[20]["degenerate"] + 2 and statistics.mean(diffs) >= -0.25 * sigma)
    print("Вывод по правилу:", "top-k 0 остаётся" if keep else
          "top-k 0 не проходит — проверяем T = 0.8 без top-k")
    print(f"отмеченных ответов {len(flagged)} — читать {folder / 'topk_flagged.txt'}")

In [ ]:
topk_compare(device=1)